#### Visulazation Check

In [ ]:
# import os
# import cv2

# # ============================================================
# # PATHS
# # ============================================================
# frames_root = "okutama-action/TestSetFrames"
# labels_dir  = "okutama-action/TestSetFrames/Labels/SingleActionLabels/3840x2160"

# DEBUG_SAVE_DIR = "debug_vis"
# os.makedirs(DEBUG_SAVE_DIR, exist_ok=True)

# # ============================================================
# # SETTINGS
# # ============================================================
# ORIG_W, ORIG_H = 3840, 2160
# DEBUG_LIMIT = 90

# # ============================================================
# # LOOP
# # ============================================================
# for drone in os.listdir(frames_root):

#     drone_path = os.path.join(frames_root, drone)
#     if not os.path.isdir(drone_path):
#         continue

#     for time_dir in os.listdir(drone_path):

#         time_path = os.path.join(drone_path, time_dir)
#         extracted_path = os.path.join(time_path, "Extracted-Frames-1280x720")

#         if not os.path.isdir(extracted_path):
#             continue

#         for video_folder in os.listdir(extracted_path):

#             video_path = os.path.join(extracted_path, video_folder)
#             label_file = os.path.join(labels_dir, f"{video_folder}.txt")

#             if not os.path.exists(label_file):
#                 continue

#             print("\n" + "="*80)
#             print(f"[VIDEO] {video_folder}")

#             # ====================================================
#             # LOAD LABELS
#             # ====================================================
#             label_lines = []
#             with open(label_file, "r") as f:
#                 for line in f:
#                     parts = line.strip().split()
#                     if len(parts) >= 10:
#                         label_lines.append(parts)

#             print("Sample frame_ids:",
#                   [int(p[5]) for p in label_lines[:10]])

#             # ====================================================
#             # SORT FRAMES (IMPORTANT)
#             # ====================================================
#             frame_files = sorted(
#                 [f for f in os.listdir(video_path) if f.endswith(".jpg")],
#                 key=lambda x: int(os.path.splitext(x)[0])
#             )

#             print("First frames:", frame_files[:5])

#             # ====================================================
#             # DEBUG LOOP
#             # ====================================================
#             for f in frame_files[:DEBUG_LIMIT]:

#                 frame_num = int(os.path.splitext(f)[0])  # 🔥 IMPORTANT

#                 img_path = os.path.join(video_path, f)
#                 img = cv2.imread(img_path)

#                 if img is None:
#                     continue

#                 h, w = img.shape[:2]

#                 print("\n--------------------------")
#                 print(f"Frame file: {f}")
#                 print(f"Frame num : {frame_num}")

#                 matched = 0

#                 for parts in label_lines:

#                     frame_id = int(parts[5])
#                     lost = int(parts[6])
#                     label_name = parts[9].strip('"')

#                     if label_name.lower() != "person":
#                         continue
#                     if lost == 1:
#                         continue

#                     # 🔥 FIX: match using frame number
#                     if frame_id != frame_num:
#                         continue

#                     matched += 1

#                     x1_raw = float(parts[1])
#                     y1_raw = float(parts[2])
#                     x2_raw = float(parts[3])
#                     y2_raw = float(parts[4])

#                     # scaling
#                     x1 = x1_raw * (w / ORIG_W)
#                     y1 = y1_raw * (h / ORIG_H)
#                     x2 = x2_raw * (w / ORIG_W)
#                     y2 = y2_raw * (h / ORIG_H)

#                     print(f"frame_id: {frame_id}")
#                     print(f"SCALED: {x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}")

#                     cv2.rectangle(img,
#                                   (int(x1), int(y1)),
#                                   (int(x2), int(y2)),
#                                   (0, 255, 0), 2)

#                 print(f"Matched boxes: {matched}")

#                 # 🔥 FIX: unique naming using frame_num
#                 out_path = os.path.join(
#                     DEBUG_SAVE_DIR,
#                     f"{video_folder}_{frame_num}.jpg"
#                 )

#                 cv2.imwrite(out_path, img)
#                 print(f"Saved: {out_path}")

#             break
#         break
#     break

#### Datset Conversion to yolo -> test

In [ ]:
# import os
# import cv2
# from tqdm import tqdm

# # ============================================================
# # PATHS
# # ============================================================
# frames_root = "okutama-action/TestSetFrames"
# labels_dir  = "okutama-action/TestSetFrames/Labels/SingleActionLabels/3840x2160"
# output_root = "dataset-yolo-person"

# # ============================================================
# # PARAMETERS
# # ============================================================
# ORIG_W, ORIG_H = 3840, 2160
# CLASS_ID = 0
# SPLIT = "test"

# # ============================================================
# # OUTPUT FOLDERS
# # ============================================================
# img_out_dir = os.path.join(output_root, "images", SPLIT)
# lbl_out_dir = os.path.join(output_root, "labels", SPLIT)

# os.makedirs(img_out_dir, exist_ok=True)
# os.makedirs(lbl_out_dir, exist_ok=True)

# # ============================================================
# # HELPER
# # ============================================================
# def convert_to_yolo(x1, y1, x2, y2, img_w, img_h):
#     bw = x2 - x1
#     bh = y2 - y1
#     cx = x1 + bw / 2
#     cy = y1 + bh / 2
#     return cx / img_w, cy / img_h, bw / img_w, bh / img_h

# # ============================================================
# # MAIN LOOP
# # ============================================================
# sample_idx = 0

# for drone in sorted(os.listdir(frames_root)):
#     drone_path = os.path.join(frames_root, drone)
#     if not os.path.isdir(drone_path):
#         continue

#     for time_dir in sorted(os.listdir(drone_path)):
#         time_path = os.path.join(drone_path, time_dir)
#         extracted_path = os.path.join(time_path, "Extracted-Frames-1280x720")

#         if not os.path.isdir(extracted_path):
#             continue

#         for video_folder in sorted(os.listdir(extracted_path)):
#             video_path = os.path.join(extracted_path, video_folder)
#             if not os.path.isdir(video_path):
#                 continue

#             label_file = os.path.join(labels_dir, f"{video_folder}.txt")
#             if not os.path.exists(label_file):
#                 continue

#             # ====================================================
#             # BUILD FRAME MAP
#             # ====================================================
#             frame_map = {}

#             with open(label_file, "r") as f:
#                 for line in f:
#                     parts = line.strip().split()
#                     if len(parts) < 10:
#                         continue

#                     try:
#                         frame_id   = int(parts[5])
#                         lost       = int(parts[6])
#                         label_name = parts[9].strip('"')
#                     except:
#                         continue

#                     if label_name.lower() != "person":
#                         continue
#                     if lost == 1:
#                         continue

#                     frame_map.setdefault(frame_id, []).append(parts)

#             # ====================================================
#             # CORRECT NUMERIC SORT
#             # ====================================================
#             frame_files = sorted(
#                 [f for f in os.listdir(video_path) if f.endswith(".jpg")],
#                 key=lambda x: int(os.path.splitext(x)[0])
#             )

#             # ====================================================
#             # LOOP FRAMES
#             # ====================================================
#             for f in tqdm(frame_files, desc=video_folder):

#                 frame_num = int(os.path.splitext(f)[0])   # 🔥 KEY FIX

#                 img_path = os.path.join(video_path, f)
#                 img = cv2.imread(img_path)
#                 if img is None:
#                     continue

#                 img_h, img_w = img.shape[:2]
#                 yolo_lines = []

#                 # ====================================================
#                 # MATCH USING frame_num (NOT idx)
#                 # ====================================================
#                 for parts in frame_map.get(frame_num, []):

#                     x1 = float(parts[1]) * (img_w / ORIG_W)
#                     y1 = float(parts[2]) * (img_h / ORIG_H)
#                     x2 = float(parts[3]) * (img_w / ORIG_W)
#                     y2 = float(parts[4]) * (img_h / ORIG_H)

#                     # clip
#                     x1 = max(0, min(x1, img_w))
#                     y1 = max(0, min(y1, img_h))
#                     x2 = max(0, min(x2, img_w))
#                     y2 = max(0, min(y2, img_h))

#                     if x2 <= x1 or y2 <= y1:
#                         continue

#                     cx, cy, bw, bh = convert_to_yolo(x1, y1, x2, y2, img_w, img_h)

#                     if bw <= 0 or bh <= 0:
#                         continue

#                     yolo_lines.append(
#                         f"{CLASS_ID} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"
#                     )

#                 # ====================================================
#                 # SKIP EMPTY (GOOD FOR TRAINING)
#                 # ====================================================
#                 if len(yolo_lines) == 0:
#                     continue

#                 # ====================================================
#                 # CORRECT NAMING
#                 # ====================================================
#                 out_name = f"{drone}_{video_folder}_{frame_num:06d}.jpg"

#                 out_img_path = os.path.join(img_out_dir, out_name)
#                 out_lbl_path = os.path.join(lbl_out_dir, out_name.replace(".jpg", ".txt"))

#                 cv2.imwrite(out_img_path, img)

#                 with open(out_lbl_path, "w") as f:
#                     f.write("\n".join(yolo_lines))

#                 sample_idx += 1

#                 if sample_idx % 1000 == 0:
#                     print(f"[INFO] Processed {sample_idx:,} samples...")

# print("\n✅ TEST conversion completed!")
# print(f"Total samples saved: {sample_idx}")

1.1.8:   0%|          | 0/1520 [00:00<?, ?it/s]

1.1.8:  76%|███████▋  | 1162/1520 [00:08<00:02, 139.70it/s]

[INFO] Processed 1,000 samples...


1.1.9:  40%|███▉      | 1055/2640 [00:07<00:12, 126.88it/s]

[INFO] Processed 2,000 samples...


...

[INFO] Processed 13,000 samples...


2.2.3:  90%|████████▉ | 1725/1922 [00:12<00:01, 111.57it/s]

[INFO] Processed 14,000 samples...


2.2.3: 100%|██████████| 1922/1922 [00:15<00:00, 127.05it/s]


✅ TEST conversion completed!
Total samples saved: 14210


#### Datset Conversion to yolo -> train

In [ ]:
# import os
# import cv2
# from tqdm import tqdm

# # ============================================================
# # PATHS
# # ============================================================
# frames_root = "okutama-action/TrainSetFrames"
# labels_dir  = "okutama-action/TrainSetFrames/Labels/SingleActionLabels/3840x2160"
# output_root = "dataset-yolo-person"

# # ============================================================
# # PARAMETERS
# # ============================================================
# ORIG_W, ORIG_H = 3840, 2160
# CLASS_ID = 0
# SPLIT = "train"

# # ============================================================
# # OUTPUT FOLDERS
# # ============================================================
# img_out_dir = os.path.join(output_root, "images", SPLIT)
# lbl_out_dir = os.path.join(output_root, "labels", SPLIT)

# os.makedirs(img_out_dir, exist_ok=True)
# os.makedirs(lbl_out_dir, exist_ok=True)

# # ============================================================
# # HELPER
# # ============================================================
# def convert_to_yolo(x1, y1, x2, y2, img_w, img_h):
#     bw = x2 - x1
#     bh = y2 - y1
#     cx = x1 + bw / 2
#     cy = y1 + bh / 2
#     return cx / img_w, cy / img_h, bw / img_w, bh / img_h

# # ============================================================
# # MAIN LOOP
# # ============================================================
# sample_idx = 0

# for drone in sorted(os.listdir(frames_root)):
#     drone_path = os.path.join(frames_root, drone)
#     if not os.path.isdir(drone_path):
#         continue

#     for time_dir in sorted(os.listdir(drone_path)):
#         time_path = os.path.join(drone_path, time_dir)
#         extracted_path = os.path.join(time_path, "Extracted-Frames-1280x720")

#         if not os.path.isdir(extracted_path):
#             continue

#         for video_folder in sorted(os.listdir(extracted_path)):
#             video_path = os.path.join(extracted_path, video_folder)
#             if not os.path.isdir(video_path):
#                 continue

#             label_file = os.path.join(labels_dir, f"{video_folder}.txt")
#             if not os.path.exists(label_file):
#                 continue

#             # ====================================================
#             # BUILD FRAME MAP
#             # ====================================================
#             frame_map = {}

#             with open(label_file, "r") as f:
#                 for line in f:
#                     parts = line.strip().split()
#                     if len(parts) < 10:
#                         continue

#                     try:
#                         frame_id   = int(parts[5])
#                         lost       = int(parts[6])
#                         label_name = parts[9].strip('"')
#                     except:
#                         continue

#                     if label_name.lower() != "person":
#                         continue
#                     if lost == 1:
#                         continue

#                     # ✅ allow generated (important)
#                     frame_map.setdefault(frame_id, []).append(parts)

#             # ====================================================
#             # CORRECT NUMERIC SORT
#             # ====================================================
#             frame_files = sorted(
#                 [f for f in os.listdir(video_path) if f.endswith(".jpg")],
#                 key=lambda x: int(os.path.splitext(x)[0])
#             )

#             # ====================================================
#             # LOOP FRAMES
#             # ====================================================
#             for f in tqdm(frame_files, desc=video_folder):

#                 frame_num = int(os.path.splitext(f)[0])  # 🔥 FIX

#                 img_path = os.path.join(video_path, f)
#                 img = cv2.imread(img_path)
#                 if img is None:
#                     continue

#                 img_h, img_w = img.shape[:2]
#                 yolo_lines = []

#                 # ====================================================
#                 # MATCH USING frame_num
#                 # ====================================================
#                 for parts in frame_map.get(frame_num, []):

#                     x1 = float(parts[1]) * (img_w / ORIG_W)
#                     y1 = float(parts[2]) * (img_h / ORIG_H)
#                     x2 = float(parts[3]) * (img_w / ORIG_W)
#                     y2 = float(parts[4]) * (img_h / ORIG_H)

#                     # clip
#                     x1 = max(0, min(x1, img_w))
#                     y1 = max(0, min(y1, img_h))
#                     x2 = max(0, min(x2, img_w))
#                     y2 = max(0, min(y2, img_h))

#                     if x2 <= x1 or y2 <= y1:
#                         continue

#                     cx, cy, bw, bh = convert_to_yolo(x1, y1, x2, y2, img_w, img_h)

#                     if bw <= 0 or bh <= 0:
#                         continue

#                     yolo_lines.append(
#                         f"{CLASS_ID} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"
#                     )

#                 # ====================================================
#                 # SKIP EMPTY (important for training)
#                 # ====================================================
#                 if len(yolo_lines) == 0:
#                     continue

#                 # ====================================================
#                 # CORRECT NAMING
#                 # ====================================================
#                 out_name = f"{drone}_{video_folder}_{frame_num:06d}.jpg"

#                 out_img_path = os.path.join(img_out_dir, out_name)
#                 out_lbl_path = os.path.join(lbl_out_dir, out_name.replace(".jpg", ".txt"))

#                 cv2.imwrite(out_img_path, img)

#                 with open(out_lbl_path, "w") as f:
#                     f.write("\n".join(yolo_lines))

#                 sample_idx += 1

#                 if sample_idx % 2000 == 0:
#                     print(f"[INFO] Processed {sample_idx:,} samples...")

# print("\n✅ TRAIN conversion completed!")
# print(f"Total samples saved: {sample_idx}")

#### Size check after conversion

In [1]:
# import os
# import cv2
# from collections import Counter

# # ====== PATH TO YOUR YOLO DATASET ROOT ======
# dataset_root = r"dataset-yolo-person"

# # ====== WHICH SPLITS TO CHECK ======
# splits = ["train", "val", "test"]
# # splits = ["test"]


# size_counter = Counter()
# total_images = 0
# bad_images = []

# for split in splits:
#     img_dir = os.path.join(dataset_root, "images", split)
#     if not os.path.exists(img_dir):
#         continue

#     print(f"\n🔍 Checking images in split: {split}")
#     for img_file in os.listdir(img_dir):
#         if not img_file.lower().endswith((".jpg", ".png")):
#             continue
#         img_path = os.path.join(img_dir, img_file)
#         img = cv2.imread(img_path)
#         if img is None:
#             bad_images.append(img_path)
#             continue
#         h, w = img.shape[:2]
#         size_counter[(w, h)] += 1
#         total_images += 1

# # ====== SUMMARY ======
# print("\n======================================================")
# print(f"📊 Total images checked: {total_images}")
# print("🖼️  Unique resolutions found:")
# for (w, h), count in size_counter.most_common():
#     print(f"   - {w}×{h} : {count} images")

# if bad_images:
#     print("\n⚠️  Failed to load these images:")
#     for b in bad_images[:10]:
#         print("   ", b)
# else:
#     print("\n✅ All images loaded successfully.")

# # ====== CONSISTENCY CHECK ======
# if len(size_counter) == 1:
#     w, h = next(iter(size_counter.keys()))
#     if (w, h) == (1280, 720):
#         print("✅ All images are consistent and correctly sized (1280×720).")
#     else:
#         print(f"⚠️ All images same size, but resolution = {w}×{h} (expected 1280×720).")
# else:
#     print("❌ Images have multiple resolutions! Check resizing pipeline.")



🔍 Checking images in split: train

🔍 Checking images in split: test

📊 Total images checked: 68874
🖼️  Unique resolutions found:
   - 1280×720 : 68874 images

✅ All images loaded successfully.
✅ All images are consistent and correctly sized (1280×720).


#### Check

In [ ]:
# import os

# def count_files_in_subdirs(path):
#     for root, dirs, files in os.walk(path):
#         # Skip the top-level directory itself, only go inside subdirs
#         if root == path:
#             continue
#         print(f"{root}: {len([f for f in files if os.path.isfile(os.path.join(root, f))])}")

# # Example usage
# directory_path = "dataset-yolo-person"
# count_files_in_subdirs(directory_path)

dataset-yolo-person/images: 0
dataset-yolo-person/images/test: 14210
dataset-yolo-person/images/train: 54664
dataset-yolo-person/labels: 2
dataset-yolo-person/labels/test: 14210
dataset-yolo-person/labels/train: 54664


#### visualizaing dataset

In [10]:
# import os
# import cv2
# import random
# import math
# import matplotlib.pyplot as plt

# # ============================================================
# # PATHS & SETTINGS
# # ============================================================
# dataset_root = "/home1/jtt_1/mtp/dataset-yolo-person"
# split = "test"   # change: "train" or "test"

# img_dir = os.path.join(dataset_root, "images", split)
# label_dir = os.path.join(dataset_root, "labels", split)

# CLASS_NAMES = ["person"]

# BOX_COLOR = (0, 255, 0)
# TEXT_COLOR = (255, 255, 255)
# BG_COLOR = (0, 0, 0)

# num_samples_to_show = 50
# show_grid = True
# grid_cols = 5

# # ============================================================
# # LOAD IMAGE FILES
# # ============================================================
# if not os.path.exists(img_dir):
#     raise FileNotFoundError(f"Image directory not found: {img_dir}")
# if not os.path.exists(label_dir):
#     raise FileNotFoundError(f"Label directory not found: {label_dir}")

# img_files = sorted([
#     f for f in os.listdir(img_dir)
#     if f.lower().endswith((".jpg", ".jpeg", ".png"))
# ])

# if len(img_files) == 0:
#     raise RuntimeError("No images found in dataset")

# random.shuffle(img_files)
# img_files = img_files[:num_samples_to_show]

# # ============================================================
# # DRAW YOLO BOXES
# # ============================================================
# def visualize_yolo_image(img_path, lbl_path):
#     img = cv2.imread(img_path)
#     if img is None:
#         print(f"[ERROR] Could not read: {img_path}")
#         return None, 0

#     img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
#     h, w = img.shape[:2]

#     box_thickness = max(2, h // 400)
#     font_scale = max(0.5, h / 1000)

#     num_objects = 0

#     if os.path.exists(lbl_path):
#         with open(lbl_path, "r") as f:
#             lines = [line.strip() for line in f if line.strip()]

#         num_objects = len(lines)

#         for line in lines:
#             parts = line.split()
#             if len(parts) != 5:
#                 continue

#             try:
#                 class_id = int(float(parts[0]))
#                 cx, cy, bw, bh = map(float, parts[1:])
#             except:
#                 continue

#             # YOLO → pixel coordinates
#             x1 = int((cx - bw / 2) * w)
#             y1 = int((cy - bh / 2) * h)
#             x2 = int((cx + bw / 2) * w)
#             y2 = int((cy + bh / 2) * h)

#             # Clamp coordinates safely
#             x1 = max(0, min(x1, w - 1))
#             y1 = max(0, min(y1, h - 1))
#             x2 = max(0, min(x2, w - 1))
#             y2 = max(0, min(y2, h - 1))

#             # Skip invalid boxes
#             if x2 <= x1 or y2 <= y1:
#                 continue

#             # Draw bounding box
#             cv2.rectangle(img, (x1, y1), (x2, y2), BOX_COLOR, box_thickness)

#             # Safe label
#             label = CLASS_NAMES[class_id] if 0 <= class_id < len(CLASS_NAMES) else f"id_{class_id}"

#             (tw, th), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, font_scale, 1)

#             # Draw label background
#             cv2.rectangle(img, (x1, y1 - th - 6), (x1 + tw + 4, y1), BG_COLOR, -1)

#             # Draw label text
#             cv2.putText(img, label, (x1 + 2, y1 - 4),
#                         cv2.FONT_HERSHEY_SIMPLEX,
#                         font_scale,
#                         TEXT_COLOR,
#                         1,
#                         cv2.LINE_AA)

#     else:
#         print(f"[WARNING] Missing label: {lbl_path}")

#     # Mark empty frames
#     if num_objects == 0:
#         cv2.putText(img, "NO OBJECTS", (20, 40),
#                     cv2.FONT_HERSHEY_SIMPLEX, 1,
#                     (255, 0, 0), 2)

#     return img, num_objects

# # ============================================================
# # VISUALIZATION
# # ============================================================
# if show_grid:

#     rows = math.ceil(len(img_files) / grid_cols)
#     plt.figure(figsize=(grid_cols * 5, rows * 4))

#     for idx, img_file in enumerate(img_files):

#         img_path = os.path.join(img_dir, img_file)
#         lbl_path = os.path.join(label_dir, os.path.splitext(img_file)[0] + ".txt")

#         print("\n" + "="*60)
#         print(f"[IMAGE] {img_path}")
#         print(f"[LABEL] {lbl_path}")

#         vis_img, num_objs = visualize_yolo_image(img_path, lbl_path)

#         print(f"[OBJECTS] {num_objs}")
#         if num_objs == 0:
#             print("[INFO] Empty label")

#         if vis_img is not None:
#             plt.subplot(rows, grid_cols, idx + 1)
#             plt.imshow(vis_img)
#             plt.title(f"{img_file}\nObjs: {num_objs}", fontsize=9)
#             plt.axis("off")

#     plt.tight_layout()
#     plt.show()

# else:
#     for img_file in img_files:

#         img_path = os.path.join(img_dir, img_file)
#         lbl_path = os.path.join(label_dir, os.path.splitext(img_file)[0] + ".txt")

#         print("\n" + "="*60)
#         print(f"[IMAGE] {img_path}")
#         print(f"[LABEL] {lbl_path}")

#         vis_img, num_objs = visualize_yolo_image(img_path, lbl_path)

#         print(f"[OBJECTS] {num_objs}")

#         if vis_img is not None:
#             plt.figure(figsize=(10, 6))
#             plt.imshow(vis_img)
#             plt.title(f"{img_file} | Objects: {num_objs}")
#             plt.axis("off")
#             plt.show()

# print(f"\nDone. Displayed {len(img_files)} samples from '{split}' set.")

In [11]:
# num_empty = 0
# num_total = 0

# for lbl in os.listdir(label_dir):
#     num_total += 1
#     if os.path.getsize(os.path.join(label_dir, lbl)) == 0:
#         num_empty += 1

# print("Empty ratio:", num_empty / num_total)

#### Visualizatioon after training

In [15]:
# from ultralytics import YOLO
# import cv2
# from matplotlib import pyplot as plt
# import random
# import os

# # --- Load model ---
# model = YOLO("/home1/jtt_1/mtp/runs-person/train/yolo8m-person/weights/best.pt")  # path to your trained model

# # --- Path to your images ---
# img_dir = "dataset-yolo-person/images/test"
# sample_imgs = random.sample(os.listdir(img_dir), 10)  # randomly pick 5 images

# for img_file in sample_imgs:
#     img_path = os.path.join(img_dir, img_file)
#     results = model.predict(source=img_path, conf=0.5, show=False, verbose=False)

#     # Visualize predictions
#     result_img = results[0].plot()  # draw bounding boxes on image

#     plt.figure(figsize=(10, 6))
#     plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
#     plt.axis("off")
#     plt.title(f"Predictions: {img_file}")
#     plt.show()


#### Saved yolo-model CSV Check

In [ ]:
# import os
# import pandas as pd

# def find_results_csv(directory):
#     """Find the results.csv file in the specified directory."""
#     for root, dirs, files in os.walk(directory):
#         if 'results.csv' in files:
#             return os.path.join(root, 'results.csv')
#     return None

# def load_results_csv(results_csv_path):
#     """Load the results CSV into a pandas DataFrame."""
#     return pd.read_csv(results_csv_path)

# def calculate_total_epochs(df):
#     """Calculate the total number of epochs from the DataFrame."""
#     return int(df['epoch'].max())

# def print_all_metrics(df):
#     """Print all available metrics from the last epoch in results.csv."""
#     final_epoch_data = df.iloc[-1]

#     print("\n========= Final Training Results =========")
#     print(f"Trained for {int(final_epoch_data['epoch'])} / {int(df['epoch'].max())} epochs")

#     print("\n--- Training Losses ---")
#     print(f" Box Loss: {final_epoch_data['train/box_loss']:.6f}")
#     print(f" Cls Loss: {final_epoch_data['train/cls_loss']:.6f}")
#     print(f" DFL Loss: {final_epoch_data['train/dfl_loss']:.6f}")

#     print("\n--- Performance Metrics ---")
#     print(f" Precision (B): {final_epoch_data['metrics/precision(B)']:.6f}")
#     print(f" Recall (B): {final_epoch_data['metrics/recall(B)']:.6f}")
#     print(f" mAP@0.5 (B): {final_epoch_data['metrics/mAP50(B)']:.6f}")
#     print(f" mAP@0.5:0.95 (B): {final_epoch_data['metrics/mAP50-95(B)']:.6f}")

#     print("\n--- Learning Rates ---")
#     print(f" LR pg0: {final_epoch_data['lr/pg0']:.8f}")
#     print(f" LR pg1: {final_epoch_data['lr/pg1']:.8f}")
#     print(f" LR pg2: {final_epoch_data['lr/pg2']:.8f}")


# def print_csv_metrics(directory):
#     """Main function to process and print all metrics from results.csv."""
#     results_csv_path = find_results_csv(directory)

#     if not results_csv_path:
#         print("Error: 'results.csv' file not found in the specified directory.")
#         return

#     print(f"[+] Found results.csv at: {results_csv_path}")

#     df = load_results_csv(results_csv_path)

#     total_epochs = calculate_total_epochs(df)
#     print(f"[+] Total number of epochs: {total_epochs}")

#     print_all_metrics(df)


# # === Run Here ===
# yolov8 = '/home1/jtt_1/mtp/runs-person/train/yolo8m-person'  # path to your YOLO run folder
# print_csv_metrics(yolov8)


[+] Found results.csv at: /home1/jtt_1/mtp/runs-person/train/yolo8m-person/results.csv
[+] Total number of epochs: 203

========= Final Training Results =========
Trained for 203 / 203 epochs

--- Training Losses ---
 Box Loss: 1.410970
 Cls Loss: 0.591370
 DFL Loss: 0.897980

--- Performance Metrics ---
 Precision (B): 0.947090
 Recall (B): 0.907230
 mAP@0.5 (B): 0.942330
 mAP@0.5:0.95 (B): 0.527100

--- Learning Rates ---
 LR pg0: 0.00195058
 LR pg1: 0.00195058
 LR pg2: 0.00195058
